In [1]:
import pandas as pd
from load_data import (
    load_bodies_from_csv
)
## Paths ##

#CSV data Paths
data_dir = "test_files"
csv_path = f"{data_dir}/TREC-07-only-phishing-6m.csv"

body_col = "body"

#Embedding paths
emb_dir = "test_embeddings"
sem_emb_path = f"{emb_dir}/semantic_embeddings.npy"
char_emb_path = f"{emb_dir}/char_embeddings.npy"
fused_emb_path = f"{emb_dir}/fused_embeddings.npy"

## Hyperparameters ##

#Semantic embeddings
model_name = "intfloat/multilingual-e5-large"

#Char (tfidf) embeddings
analyzer = "char_wb",
ngram_range = (3, 5),
min_df = 3, #minimum amount of documents a token must appear in
max_features = 200_000 #max n_grams kept, keeps the features with the highest document frequency
dim_reduction = 256
seed = 42

#Test
bodies = load_bodies_from_csv(csv_path=csv_path, body_col=body_col)
print(bodies[0])









Do you feel the pressure to perform and not rising to the occasion??




Try Viagra.....
your anxiety will be a thing of the past and you will
be back to your old self.




In [ ]:
# --- Text embeddings (semantic + char-style) -> save to ./test_embeddings/ ---

import os
import json
import numpy as np

# Now import your functions
from text_embeddings import (
    get_semantic_embeddings,
    train_char_tfidf_model,
    train_tfidf_svd_reducer,
    get_char_embeddings,
    fuse_text_embeddings,
)

# 1) Semantic embeddings (E5)
E_sem = get_semantic_embeddings(bodies, batch_size=32, device=None, model_name=model_name)  # numpy (N, 1024)

# 2) Char-style embeddings (TF-IDF char n-grams + SVD)
# NOTE: For this "test" step we fit on all bodies. Later, fit ONLY on train split.
tfidf_model, X_train = train_char_tfidf_model(
    bodies,
    ngram_range=ngram_range,
    min_df=min_df,
    max_features=max_features,
)

# Reduced via LSA, which works better than PCA for sparse vectors
svd_reducer = train_tfidf_svd_reducer(X_train, out_dim=dim_reduction, seed=seed)

E_char = get_char_embeddings(bodies, tfidf_model=tfidf_model, svd_reducer=svd_reducer, l2_normalize=True)  # numpy (N, 256)

# 3) Fuse
E_text = fuse_text_embeddings(E_sem, E_char)  # numpy (N, 1280)

# 4) Save embeddings
np.save(sem_emb_path, E_sem)
np.save(char_emb_path, E_char)
np.save(fused_emb_path, E_text)

# 5) Save config/metadata for reproducibility
meta = {
    "sentence_model": f"{model_name}",
    "semantic_dim": int(E_sem.shape[1]),
    "tfidf": {
        "analyzer": "char_wb",
        "ngram_range": ngram_range,
        "min_df": min_df,
        "max_features": max_features,
    },
    "svd": {
        "out_dim": dim_reduction,
        "seed": seed,
    },
    "fused_dim": int(E_text.shape[1]),
    "num_emails": int(E_text.shape[0]),
}
with open(f"{emb_dir}/metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"Saved embeddings to ./{emb_dir}/")
print("E_sem:", E_sem.shape, "E_char:", E_char.shape, "E_text:", E_text.shape)


Batches:   0%|          | 0/885 [00:00<?, ?it/s]

Saved embeddings to ./test_embeddings/
E_sem: (28305, 1024) E_char: (28305, 256) E_text: (28305, 1280)
